In [ ]:
!nvidia-smi

In [ ]:
import os 
os.environ["CUDA_VISIBLE_DEVICES"] = "1"

In [ ]:
from transformers import (
    AutoModel,
    AutoModelForCausalLM,
    AutoConfig,
    AutoProcessor,
    AutoTokenizer,
    RobertaConfig,
)

import torch
import torch.nn as nn

from parser import (remove_comments_and_docstrings,
                   tree_to_token_index,
                   index_to_code_token,
                   tree_to_variable_index)
from tree_sitter import Language, Parser
from preprocess import AST

from modeling_llava_code import LlavaCodeConfig,  LlavaCodeForConditionalGeneration

from huggingface_hub import login
from dotenv import load_dotenv

load_dotenv()
token = os.getenv("HF_TOKEN")
login(token=token)

device = torch.device("cuda:0")

In [ ]:
from dataclasses import dataclass

@dataclass
class ArgsMock:
    text_model_id = "bigcode/starcoderbase-1b"
    structure_model_id = "microsoft/unixcoder-base"

args = ArgsMock()

In [ ]:
structure_config = RobertaConfig.from_pretrained(args.structure_model_id)
structure_config.model_id = args.structure_model_id
text_config = AutoConfig.from_pretrained(args.text_model_id)
text_config.model_id = args.text_model_id
configuration = LlavaCodeConfig(structure_config, text_config, structure_token_od=25782)
configuration

In [ ]:
structure_config

In [ ]:
text_config

In [ ]:
configuration.output_attentions, configuration.output_hidden_states, configuration.use_return_dict

In [ ]:
model = LlavaCodeForConditionalGeneration(configuration).to(device)

In [ ]:
model

In [ ]:
processor = AutoProcessor.from_pretrained(args.text_model_id)
prompt = 'with open('
inputs = processor(prompt, return_tensors="pt").to(device)
inputs

In [ ]:
output = model.generate(**inputs, max_new_tokens=5)

In [ ]:
output

In [ ]:
model

In [ ]:
processor.decode(output[0])

In [ ]:
# state_dict = AutoModelForCausalLM.from_pretrained(args.text_model_id).state_dict()
# state_dict = {key.replace('transformer.', 'model.language_model.'): value for (key, value) in state_dict.items()}
# model.load_state_dict(state_dict, strict=False)

In [ ]:
# output = model.generate(**inputs, max_new_tokens=5)

In [ ]:
# processor.decode(output[0])

In [ ]:
prompt = 'with open('

In [ ]:
print(prompt)
ast = AST(prompt, 'python', model.model.structure_model.tokenizer)
max_length = 512
tokens = ast[:max_length-4]
tokens = [model.model.structure_model.tokenizer.cls_token, "<encoder-only>", model.model.structure_model.tokenizer.sep_token] + tokens + [model.model.structure_model.tokenizer.sep_token]
ast_ids = model.model.structure_model.tokenizer.convert_tokens_to_ids(tokens)
ast_ids = torch.tensor(ast_ids).unsqueeze(0).to(device)

In [ ]:
tokens, ast_ids

In [ ]:
prompt = 'with open('
print(prompt)
ast = AST(prompt, 'python', model.model.structure_model.tokenizer)
max_length = 512
tokens = ast[:max_length-4]
tokens = [model.model.structure_model.tokenizer.cls_token, "<encoder-only>", model.model.structure_model.tokenizer.sep_token] + tokens + [model.model.structure_model.tokenizer.sep_token]
ast_ids = model.model.structure_model.tokenizer.convert_tokens_to_ids(tokens)
ast_ids = torch.tensor(ast_ids).unsqueeze(0).to(device)

prompt += '>:<'
inputs = processor(prompt, return_tensors="pt").to(device)
assert inputs.input_ids[-1][-1].item() == model.config.structure_token_id
output = model.generate(**inputs, structure_values=ast_ids, max_new_tokens=5)

In [ ]:
processor.decode(output[0])

In [ ]:
# inputs = processor(prompt, return_tensors="pt")
# input_ids = torch.cat([inputs.input_ids, torch.tensor([[-100]]).to(inputs.input_ids.dtype)], axis=-1).to(device)
# attention_mask = torch.cat([inputs.attention_mask, torch.tensor([[1]]).to(inputs.attention_mask.dtype)], axis=1).to(device)
# output = model.generate(input_ids=input_ids, attention_mask=attention_mask, structure_values=ast_ids, max_new_tokens=5)

In [ ]:
input_ids, attention_mask

In [ ]:
model

In [ ]:
!nvidia-smi